In [ ]:
from pathlib import Path
from torch.utils.data import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
import pandas as pd, numpy as np, torch, os

In [ ]:
BN_SOURCE_DIR = Path(r"D:\MSc\3. Spring 2026\CSE715\Project\bn_raw_dataset")
BN_SOURCE_DIR.exists()

In [ ]:
TRAINING_DATA_PATH = BN_SOURCE_DIR / "BanglaSongLyrics.csv"
UNSEEN_DATA_PATH = BN_SOURCE_DIR / "transcripts.csv"

In [ ]:
OUTPUT_DIR = BN_SOURCE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
BANGLABERT = OUTPUT_DIR / "BanglaBERT_genre_2"
BANGLABERT.mkdir(exist_ok=True, parents=True)

In [ ]:
MAX_LEN    = 512
BATCH_SIZE = 16
EPOCHS     = 10
LR         = 1e-5
SEED       = 42

In [ ]:
BANGLABERT_MODEL = "csebuetnlp/banglabert"

In [ ]:
df = pd.read_csv(TRAINING_DATA_PATH)
print(f"Columns: {df.columns.tolist()}")
print(f"Total samples: {len(df)}")
print(f"Genre distribution:\n{df['category'].value_counts()}")

In [ ]:
df = df.dropna(subset=['lyrics', 'category'])
df = df[df['category'] != 'Uncategorized']
df = df.groupby('category').filter(lambda x: len(x) >= 100)
df['lyrics'] = df['lyrics'].astype(str).str.strip()
df = df[df['lyrics'] != '']

In [ ]:
from sklearn.utils import resample

df_list = []
max_size = 500  # Cap the majority at 500 or boost minority to 500

for category in df['category'].unique():
    df_category = df[df['category'] == category]
    if len(df_category) < max_size:
        # Upsample minority
        df_category_upsampled = resample(df_category, 
                                         replace=True, 
                                         n_samples=max_size, 
                                         random_state=42)
        df_list.append(df_category_upsampled)
    else:
        # Downsample majority slightly to reduce bias
        df_category_downsampled = resample(df_category, 
                                           replace=False, 
                                           n_samples=max_size, 
                                           random_state=42)
        df_list.append(df_category_downsampled)

balanced_df = pd.concat(df_list)

In [ ]:
df_copy = df.copy(deep=True)

In [ ]:
df = balanced_df

In [ ]:
df['category'].value_counts()

In [ ]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['category'])
genre_names = le.classes_.tolist()
num_classes = len(genre_names)
print(f"\nGenres ({num_classes}): {genre_names}")

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=SEED)
val_df,  test_df  = train_test_split(test_df, test_size=0.5, stratify=test_df['label'], random_state=SEED)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
class LyricsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # Much cleaner: No squeezing needed because we tokenized beforehand
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}


def train_model(model_name, output_dir, train_df, val_df, test_df, num_classes, genre_names, max_len=MAX_LEN):

    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")

    # Tokenizer and datasets
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_data(df):
        return tokenizer(df['lyrics'].tolist(), truncation=True, padding='max_length', max_length=MAX_LEN)

    train_enc = tokenize_data(train_df)
    val_enc   = tokenize_data(val_df)
    test_enc  = tokenize_data(test_df)

    train_ds = LyricsDataset(train_enc, train_df['label'].tolist())
    val_ds   = LyricsDataset(val_enc, val_df['label'].tolist())
    test_ds  = LyricsDataset(test_enc, test_df['label'].tolist())

    # Model
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_classes,
        id2label={i: g for i, g in enumerate(genre_names)},
        label2id={g: i for i, g in enumerate(genre_names)}
    )

    # Training arguments
    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LR,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        logging_dir=f"{output_dir}/logs",
        logging_steps=50,
        seed=SEED,
        fp16=torch.cuda.is_available(),
        report_to="none",
        remove_unused_columns=False
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    # Train
    trainer.train()

    # Evaluate on test set
    print(f"\n── Test Set Evaluation: {model_name} ──")
    test_preds_out = trainer.predict(test_ds)
    test_preds     = np.argmax(test_preds_out.predictions, axis=1)
    print(classification_report(test_df['label'].tolist(), test_preds,
                                  target_names=genre_names))

    # Save best model and tokenizer
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Model saved to {output_dir}")

    return trainer, tokenizer

## WARNING: This is VERY time-consuming.

In [ ]:
# Train BanglaBERT
bb_trainer, bb_tokenizer = train_model(
    BANGLABERT_MODEL, BANGLABERT,
    train_df, val_df, test_df,
    num_classes, genre_names
)

In [ ]:
def predict_on_third_dataset(model_dir, third_csv, genre_names,
                              le, output_csv, max_len=MAX_LEN):

    print(f"\n── Inference with {model_dir} ──")

    # Load saved model
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model     = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Load third dataset
    third_df = pd.read_csv(third_csv)
    third_df  = third_df.dropna(subset=['sentence'])
    third_df['sentence'] = third_df['sentence'].astype(str).str.strip()

    # Predict in batches
    all_preds  = []
    all_probs  = []
    batch_size = 32

    for i in range(0, len(third_df), batch_size):
        batch_texts = third_df['sentence'].iloc[i:i+batch_size].tolist()
        encoding = tokenizer(
            batch_texts,
            max_length=max_len,
            padding=True,
            truncation=True,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            logits = model(**encoding).logits
            probs  = torch.softmax(logits, dim=1).cpu().numpy()
            preds  = np.argmax(probs, axis=1)

        all_preds.extend(preds)
        all_probs.extend(probs.max(axis=1))   # confidence of top prediction

        if i % 320 == 0:
            print(f"  Processed {min(i+batch_size, len(third_df))}/{len(third_df)}")

    third_df['predicted_genre']      = [genre_names[p] for p in all_preds]
    third_df['confidence']           = all_probs
    third_df['predicted_label_idx']  = all_preds

    third_df.to_csv(output_csv, index=False)
    print(f"Saved to {output_csv}")
    print(f"\nGenre distribution:\n{third_df['predicted_genre'].value_counts()}")
    print(f"Mean confidence: {np.mean(all_probs):.4f}")

    return third_df

In [ ]:
# Run inference with both models
bb_results  = predict_on_third_dataset(
    BANGLABERT, UNSEEN_DATA_PATH, genre_names, le,
    "unseen_data_predictions_banglaBERT.csv"
)

In [ ]:
def unify_song_genres(results_df):
    """
    The unseen dataset has segments from the same audio file, which have been classified under different genres. This makes the genre assignment of the segments belonging to the same audio file the same.
    """
    # 1. Extract the base Audio ID
    results_df['audio_id'] = results_df['id'].str.split('_seg').str[0]
    
    # 2. Calculate the weighted winner for each audio_id
    # We sum confidence scores per genre for each song
    weighted_votes = results_df.groupby(['audio_id', 'predicted_genre'])['confidence'].sum().reset_index()
    
    # 3. Find the winning genre string for each audio_id
    # We sort by confidence and take the top one for each ID
    winners = weighted_votes.sort_values(['audio_id', 'confidence'], ascending=[True, False]) \
                            .drop_duplicates('audio_id')
    
    # Create a mapping dictionary: { 'audio_id': 'winning_genre' }
    winner_map = dict(zip(winners['audio_id'], winners['predicted_genre']))
    
    # 4. Assign the winning genre to ALL segments of that song
    # This keeps the original row count but "fixes" the inconsistent labels
    results_df['unified_genre'] = results_df['audio_id'].map(winner_map)
    
    # Optional: Do the same for a "unified_confidence" (the total sum of the winning genre)
    confidence_map = dict(zip(winners['audio_id'], winners['confidence']))
    results_df['unified_total_confidence'] = results_df['audio_id'].map(confidence_map)
    
    return results_df

# Usage:
bb_results_updated = unify_song_genres(bb_results)
print(bb_results_updated[['id', 'predicted_genre', 'unified_genre', 'confidence']].head(20))

In [ ]:
previous_df = pd.read_csv("third_dataset_predictions_banglaBERT.csv")
previous_unified_df = unify_song_genres(previous_df)
print(previous_unified_df[['id', 'predicted_genre', 'unified_genre', 'confidence']].head(20))

In [ ]:
previous_unified_df.to_csv("metadata_bn_unified.csv", index=False)